<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA

### DATASET GENERATOR and Liq ML Parameters


In [ ]:
N        = 40          # grid points in x,y (Q, T, f are NxN)
Lx, Ly   = 0.05, 0.05
rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
T_m      = 330.0
T_init   = 300.0
T_bound  = 330.0
t_end    = 4000.0
save_times = (100.0, 250.0, 400.0, 600.0, 1000.0, 1200.0, 1500.0, 1800.0, 2100.0)
cfl      = 0.45

# Heat-source GP
q_scale        = 1e5     # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# Boundary Conditions
mu_max=0.8
sigma_max=0.7      
amp_max=80

# Dataset sizes
NUM_CASES      = 200.0      # total function-realizations / cases
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1      # test is the remainder

# DeepONet sampling
sensor_mode    = "full"   # "full" (flatten full NxN) or "downsample"
S_down         = 40       # used if sensor_mode="downsample" (S_down x S_down grid)
n_time_bc      = len(save_times)
S_down_BC      = N

points_per_case_per_time = N**2  # # of (x,y) points sampled per time snapshot

# Boundary configuration control
only_lr_vary   = True     # True: top & bottom constant, left/right varied; False: allow all configurable
All_side_const_temp_boundary = False    # if all sides constant boundary is wanted at T_bound for dataset testing

# MODEL PARAMETERS

BATCH_POINTS = 32768//2
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4# first case 0.0
VAL_SAMPLES = 4
VAL_BATCH_POINTS = 32768//2
STEPS_PER_EPOCH = 64

#each path weight
alpha_add=1.0
alpha_prod=0.2

### LIQUID FRACTION PREDICTIONS

In [ ]:
import glob
# -----------------------------
# 0) Config / Auto-run directory
# -----------------------------
ROOT = Path(".")
# hardcoded one: RUN_DIR = Path("./dataset_run_YYYYMMDD-HHMMSS")
def _latest_run_dir(root: Path) -> Path | None:
    cands = sorted([Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()])
    return cands[-1] if cands else None

RUN_DIR = _latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Build dataset first."

DATA_FRAC = RUN_DIR / "deeponet_frac_dataset.npz"
DATA_TEMP = RUN_DIR / "deeponet_temp_dataset.npz"
DATA_SPLIT = RUN_DIR / "splits.json"

print(f"Using RUN_DIR: {RUN_DIR}")

In [ ]:
# -----------------------------
# 1) Load dataset + splits
# -----------------------------
D = np.load(DATA_FRAC, allow_pickle=True)
D2 = np.load(DATA_TEMP, allow_pickle=True)
# ---- per-case branch features ----
branch_T   = D["branch_T"]                  # [C, S_Tsnap]  temperature snapshots for frac
branch_Q   = D2["branch_Q"]                 # [C, S_Q_all]  heat-source field Q from temp dataset
branch_BC  = D2["branch_BC"]                # [C, S_BC]     boundary conditions from temp dataset

# sanity check: same number of cases in both datasets
C_frac = branch_T.shape[0]
C_temp_Q  = branch_Q.shape[0]
C_temp_BC = branch_BC.shape[0]
assert C_frac == C_temp_Q == C_temp_BC, \
    f"Case count mismatch: frac={C_frac}, temp_Q={C_temp_Q}, temp_BC={C_temp_BC}"

# ---- point-level trunk + targets from frac dataset ----
trunk_xy   = D["trunk_xy"]                  # [M, 2]
trunk_t    = D["trunk_t"]                   # [M, 1]
yf         = D["yf"].reshape(-1, 1)         # [M, 1] liquid fraction
case_ids   = D["case_ids"].astype(np.int64) # [M]   (case index for each point)
meta       = D["meta"].item()

N          = int(meta["N"])
Lx, Ly     = float(meta["Lx"]), float(meta["Ly"])
save_times = np.array(meta["save_times"], dtype=float)

print(f"Loaded FRAC : branch_T{branch_T.shape}, "
      f"trunk_xy{trunk_xy.shape}, trunk_t{trunk_t.shape}, yf{yf.shape}")
print(f"Loaded TEMP : branch_Q{branch_Q.shape}, branch_BC{branch_BC.shape}")

# ---- case-level splits (same style as before) ----
DATA_SPLIT = Path(DATA_SPLIT)  # ensure Path
if DATA_SPLIT.exists():
    splits = json.loads(Path(DATA_SPLIT).read_text())
    train_ids = np.array(splits["train"], dtype=int)
    val_ids   = np.array(splits["val"],   dtype=int)
    test_ids  = np.array(splits["test"],  dtype=int)
else:
    C = branch_T.shape[0]
    idx = np.arange(C)
    np.random.default_rng(2024).shuffle(idx)
    n_tr  = int(round(0.8 * C))
    n_val = int(round(0.1 * C))
    train_ids = idx[:n_tr]
    val_ids   = idx[n_tr:n_tr + n_val]
    test_ids  = idx[n_tr + n_val:]
    splits = {
        "train": train_ids.tolist(),
        "val":   val_ids.tolist(),
        "test":  test_ids.tolist()
    }
    Path(DATA_SPLIT).write_text(json.dumps(splits, indent=2))
    print("splits.json not found → wrote default 80/10/10 split.")

print(f"Split sizes → train:{len(train_ids)}  val:{len(val_ids)}  test:{len(test_ids)}")


In [ ]:
#checker
print("S_T_all:", branch_T.shape[1])
print("Example rows:",
      "bT:", branch_T[0,:5],
      "xy:", trunk_xy[:2],
      "t:", trunk_t[:2].ravel())

In [ ]:
def _mask_from_cases(allowed_case_ids: np.ndarray) -> np.ndarray:
    allowed = np.zeros(branch_T.shape[0], dtype=bool)
    allowed[allowed_case_ids] = True
    return allowed[case_ids]


mask_tr = _mask_from_cases(train_ids)
mask_val = _mask_from_cases(val_ids)
mask_te  = _mask_from_cases(test_ids)

# -------------------------
# point-level tensors
# -------------------------
xy_tr = trunk_xy[mask_tr]        # [M_tr, 2]
t_tr  = trunk_t[mask_tr]         # [M_tr, 1]
y_tr  = yf[mask_tr]              # [M_tr, 1]
y_mean = float(y_tr.mean())
y_std  = float(y_tr.std() + 1e-6)
cid_tr = case_ids[mask_tr]

xy_val = trunk_xy[mask_val]
t_val  = trunk_t[mask_val]
y_val  = yf[mask_val]
cid_val = case_ids[mask_val]

# ============================================================
# branch-level stats (TRAIN CASES ONLY)
# ============================================================

# ----- Temperature T-snapshot branch -----
bT_tr   = branch_T[train_ids]               # [C_tr, S_Tsnap]
bT_mean = bT_tr.mean(axis=0, keepdims=True)
bT_std  = bT_tr.std(axis=0, keepdims=True) + 1e-8

# ----- Q branch -----
bQ_tr   = branch_Q[train_ids]               # [C_tr, S_Q_all]
bQ_mean = bQ_tr.mean(axis=0, keepdims=True)
bQ_std  = bQ_tr.std(axis=0, keepdims=True) + 1e-8

# ----- BC branch -----
bBC_tr   = branch_BC[train_ids]             # [C_tr, S_BC]
bBC_mean = bBC_tr.mean(axis=0, keepdims=True)
bBC_std  = bBC_tr.std(axis=0, keepdims=True) + 1e-8

# ============================================================
# coordinate + time normalization
# ============================================================
xy_min = np.array([0.0, 0.0], dtype=np.float32)
xy_max = np.array([Lx, Ly], dtype=np.float32)
t_min  = float(save_times.min())
t_max  = float(save_times.max())


# ============================================================
# Normalization functions
# ============================================================

def norm_y(y):
    return ((y - y_mean) / y_std).astype(np.float32)

def denorm_y(y_hat):
    # y_hat is torch tensor
    return y_hat * y_std + y_mean


def norm_branch_T(bT):
    return ((bT - bT_mean) / bT_std).astype(np.float32)

def norm_branch_Q(bQ):
    return ((bQ - bQ_mean) / bQ_std).astype(np.float32)

def norm_branch_BC(bBC):
    return ((bBC - bBC_mean) / bBC_std).astype(np.float32)


def norm_xy(xy):
    return ((xy - xy_min) / np.maximum(xy_max - xy_min, 1e-6)).astype(np.float32)

def norm_t(tt):
    return ((tt - t_min) / max(t_max - t_min, 1e-6)).astype(np.float32)


In [ ]:
# -----------------------------
# 3) Dataset for pooled point sampling (5-stream: Q, BC, T, xy, t)
# -----------------------------
class PooledPointDataset5(Dataset):
    def __init__(self, trunk_xy, trunk_t, yf, case_ids,
                 branch_Q, branch_BC, branch_T, allowed_case_ids,
                 batch_points=65536):
        self.xy       = trunk_xy
        self.tt       = trunk_t
        self.y        = yf.astype(np.float32)
        self.case_ids = case_ids

        # per-case branch data
        self.branch_Q  = branch_Q
        self.branch_BC = branch_BC
        self.branch_T  = branch_T

        # which cases are allowed in this split (train/val/test)
        self.allowed = np.zeros(self.branch_T.shape[0], dtype=bool)
        self.allowed[allowed_case_ids] = True

        # rows (point indices) whose case_ids are in allowed_case_ids
        self.rows = np.where(self.allowed[self.case_ids])[0]   # indices eligible

        self.batch_points = int(batch_points)

        # pre-normalize trunk
        self.xy_n = norm_xy(self.xy)   # [M, 2]
        self.tt_n = norm_t(self.tt)    # [M, 1]

    def __len__(self):
        return 10_000_000  # virtual

    def __getitem__(self, idx):
        # sample random point indices from the eligible rows
        ridx = np.random.randint(0, self.rows.shape[0], size=(self.batch_points,))
        rows = self.rows[ridx]

        xy   = self.xy_n[rows]           # [B, 2]
        tt   = self.tt_n[rows]           # [B, 1]
        y    = self.y[rows]              # [B, 1]
        y_n  = norm_y(y)                 # normalized target

        cids = self.case_ids[rows]       # [B]

        # normalize branch features per point (by case)
        bQ   = norm_branch_Q(self.branch_Q[cids])    # [B, S_Q_all]
        bBC  = norm_branch_BC(self.branch_BC[cids])  # [B, S_BC]
        bT   = norm_branch_T(self.branch_T[cids])    # [B, S_Tsnap]

        return (torch.from_numpy(bQ),
                torch.from_numpy(bBC),
                torch.from_numpy(bT),
                torch.from_numpy(xy),
                torch.from_numpy(tt),
                torch.from_numpy(y_n))


In [ ]:
# -----------------------------
# 4) Model: 5-network DeepONet with dot-product head
#      Branch: Q, BC, T
#      Trunk : (x,y), t
# -----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, act=nn.GELU, dropout=0.1):
        super().__init__()
        layers = []
        dims = (in_dim,) + tuple(hidden) + (out_dim,)
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), act(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
    def forward(self, x): 
        return self.net(x)


class DeepONet5(nn.Module):
    """
    y(x,y,t | Q, BC, T) = α_concat * f_concat(Q,BC,T,xy,t)
                        + α_prod   * f_prod(Q,BC,T,xy,t)
                        + b

    where:
      - 5 subnetworks each output a D-dim embedding:
            branchQ  : Q-field features
            branchBC : boundary-condition features
            branchT  : temperature-snapshot features
            trunkXY  : spatial (x,y)
            trunkT   : time t
      - f_concat uses concatenation of all 5 embeddings and an expressive MLP head.
      - f_prod   uses elementwise product of all 5 embeddings, summed and scaled.
    """
    def __init__(self,
                 S_Q_all,          # length of Q feature vector
                 S_BC,             # length of BC feature vector
                 S_T_all,          # length of T-snapshot feature vector
                 D=256,
                 Q_hidden=(256,256,256),
                 BC_hidden=None,
                 T_hidden=(256,256,256),
                 xy_hidden=(256,256,256,256),
                 t_hidden=(128,128)):
        super().__init__()

        if BC_hidden is None:
            BC_hidden = Q_hidden

        # ---------------- Branch networks ----------------
        self.branchQ  = MLP(S_Q_all, Q_hidden,  D)
        self.branchBC = MLP(S_BC,    BC_hidden, D)
        self.branchT  = MLP(S_T_all, T_hidden,  D)

        # ---------------- Trunk networks -----------------
        self.trunkXY  = MLP(2,       xy_hidden, D)
        self.trunkT   = MLP(1,       t_hidden,  D)

        # ---------------- LayerNorms ---------------------
        self.lnQ   = nn.LayerNorm(D)
        self.lnBC  = nn.LayerNorm(D)
        self.lnT   = nn.LayerNorm(D)
        self.lnXY  = nn.LayerNorm(D)
        self.lnTau = nn.LayerNorm(D)

        # scaling for multiplicative path
        self.scale5 = (D ** 0.5)

        # ---------------- Expressive head ----------------
        # concatenation of 5 embeddings → 5D
        self.head = MLP(in_dim=5*D, hidden=(256,128), out_dim=1, act=nn.GELU)

        # tiny learned scalars to weight each path
        self.alpha_concat = nn.Parameter(torch.tensor(alpha_add))
        self.alpha_prod   = nn.Parameter(torch.tensor(alpha_prod))
        self.bias         = nn.Parameter(torch.zeros(1))

    def forward(self, bQ, bBC, bT, xy, tt):
        """
        bQ  : [B, S_Q_all]
        bBC : [B, S_BC]
        bT  : [B, S_T_all]
        xy  : [B, 2]
        tt  : [B, 1]
        """
        # embeddings + layer norms
        eQ   = self.lnQ(self.branchQ(bQ))       # [B, D]
        eBC  = self.lnBC(self.branchBC(bBC))    # [B, D]
        eT   = self.lnT(self.branchT(bT))       # [B, D]
        eXY  = self.lnXY(self.trunkXY(xy))      # [B, D]
        eTau = self.lnTau(self.trunkT(tt))      # [B, D]

        # ----- expressive additive path (concat) -----
        z_concat = torch.cat([eQ, eBC, eT, eXY, eTau], dim=1)  # [B, 5D]
        y_concat = self.head(z_concat)                         # [B, 1]

        # ----- multiplicative residual path -----
        e_prod = eQ * eBC * eT * eXY * eTau                    # [B, D]
        y_prod = e_prod.sum(dim=1, keepdim=True) / self.scale5 # [B, 1]

        # ----- combine -----
        yhat_norm = self.alpha_concat * y_concat + self.alpha_prod * y_prod + self.bias
        return yhat_norm


In [ ]:
import torch, importlib
print(torch.__version__)
importlib.import_module("torch._utils")
import torch.optim as optim
optim.Adam([torch.nn.Parameter(torch.randn(2,requires_grad=True))], lr=1e-3)
print("Adam OK")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")

In [ ]:
# -----------------------------
# 5) Train / Validate
# -----------------------------

S_Q_all  = branch_Q.shape[1]
S_BC     = branch_BC.shape[1]
S_T_all  = branch_T.shape[1]

model = DeepONet5(S_Q_all=S_Q_all,
                  S_BC=S_BC,
                  S_T_all=S_T_all,
                  D=256,
                  Q_hidden=(256,256,256),
                  BC_hidden=None,
                  T_hidden=(256,256,256),
                  xy_hidden=(256,256,256,256),
                  t_hidden=(128,128)).to(device)

BATCH_POINTS     = BATCH_POINTS
VAL_BATCH_POINTS = VAL_BATCH_POINTS
EPOCHS           = EPOCHS
LR               = LR
WEIGHT_DECAY     = WEIGHT_DECAY
VAL_SAMPLES      = VAL_SAMPLES
STEPS_PER_EPOCH  = STEPS_PER_EPOCH

# we use the full arrays trunk_xy, trunk_t, yf, case_ids
# and let allowed_case_ids control which cases are sampled
ds_tr = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                            branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                            allowed_case_ids=train_ids,
                            batch_points=BATCH_POINTS)

ds_va = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                            branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                            allowed_case_ids=val_ids,
                            batch_points=VAL_BATCH_POINTS)

ds_tr.batch_points = BATCH_POINTS
ds_va.batch_points = VAL_BATCH_POINTS

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_val = math.inf
ckpt_path = RUN_DIR / "deeponet_frac_model_5net.pt"

print("Starting training…")
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss_acc = 0.0
    train_rmse_acc = 0.0

    for _ in range(STEPS_PER_EPOCH):
        bQ, bBC, bT, xy, tt, ytrue_n = ds_tr[0]
        bQ = bQ.to(device)
        bBC = bBC.to(device)
        bT = bT.to(device)
        xy = xy.to(device)
        tt = tt.to(device)
        ytrue_n = ytrue_n.to(device)

        opt.zero_grad(set_to_none=True)
        yhat_n = model(bQ, bBC, bT, xy, tt)
        loss = ((yhat_n - ytrue_n)**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            # RMSE in physical space (here: liquid fraction units)
            yhat_phys = denorm_y(yhat_n)
            y_phys    = denorm_y(ytrue_n)
            train_rmse_acc += torch.sqrt(((yhat_phys - y_phys)**2).mean()).item()
            train_loss_acc += loss.item()

    # ----------------- validation -----------------
    model.eval()
    with torch.no_grad():
        vloss_acc = 0.0
        rmse_acc  = 0.0
        for _ in range(VAL_SAMPLES):
            bQv, bBCv, bTv, xyv, ttv, yv_n = ds_va[0]
            bQv = bQv.to(device)
            bBCv = bBCv.to(device)
            bTv = bTv.to(device)
            xyv = xyv.to(device)
            ttv = ttv.to(device)
            yv_n = yv_n.to(device)

            yhatv_n = model(bQv, bBCv, bTv, xyv, ttv)
            vloss_acc += ((yhatv_n - yv_n)**2).mean().item()

            yhatv_phys = denorm_y(yhatv_n)
            yv_phys    = denorm_y(yv_n)
            rmse_acc += torch.sqrt(((yhatv_phys - yv_phys)**2).mean()).item()

    train_mse_n  = train_loss_acc / STEPS_PER_EPOCH
    train_rmse   = train_rmse_acc / STEPS_PER_EPOCH
    vloss        = vloss_acc / VAL_SAMPLES
    val_rmse     = rmse_acc / VAL_SAMPLES

    print(f"alphas → concat={model.alpha_concat.item():.4f} | prod={model.alpha_prod.item():.4f}")
    print(f"Epoch {epoch:03d} | train MSE(n){train_mse_n:.6e} | val MSE(n){vloss:.6e} "
          f"| train RMSE{train_rmse:.3f} | val RMSE{val_rmse:.3f}")

    if vloss < best_val:
        best_val = vloss
        torch.save({
            "model": model.state_dict(),
            "bQ_mean": bQ_mean,  "bQ_std":  bQ_std,
            "bBC_mean": bBC_mean, "bBC_std": bBC_std,
            "bT_mean": bT_mean,  "bT_std":  bT_std,
            "xy_min": xy_min,    "xy_max":  xy_max,
            "t_min": t_min,      "t_max":   t_max,
            "y_mean": y_mean,    "y_std":   y_std,
            "S_Q_all": S_Q_all,
            "S_BC": S_BC,
            "S_T_all": S_T_all,
            "meta": meta,
        }, ckpt_path)
        print("  -> checkpoint saved.")

print(f"Best val MSE(n): {best_val:.6e}")
print(f"Saved checkpoint → {ckpt_path}")


In [ ]:
#-------------------------------
# 6) INFERENCE PLOTS (liq fraction)
#-------------------------------

@torch.no_grad()
def load_trained(path: Path):
    ck = torch.load(path, map_location="cpu", weights_only=False)

    # Shapes from checkpoint
    S_Q_all = ck.get("S_Q_all", None)
    S_BC    = ck.get("S_BC", None)
    S_T_all = ck.get("S_T_all", None)
    assert S_Q_all is not None and S_BC is not None and S_T_all is not None, \
        "Checkpoint missing S_Q_all / S_BC / S_T_all"

    # Instantiate DeepONet5 with same defaults used at train time
    m = DeepONet5(S_Q_all=S_Q_all,
                  S_BC=S_BC,
                  S_T_all=S_T_all,
                  D=256,
                  Q_hidden=(256,256,256),
                  BC_hidden=None,
                  T_hidden=(256,256,256),
                  xy_hidden=(256,256,256,256),
                  t_hidden=(128,128)).to(device)
    m.load_state_dict(ck["model"], strict=True)
    m.eval()

    # Stats for normalization / denormalization
    stats = {
        "bQ_mean": ck["bQ_mean"],  "bQ_std":  ck["bQ_std"],
        "bBC_mean": ck["bBC_mean"], "bBC_std": ck["bBC_std"],
        "bT_mean": ck["bT_mean"],  "bT_std":  ck["bT_std"],
        "xy_min": ck["xy_min"],    "xy_max":  ck["xy_max"],
        "t_min": ck["t_min"],      "t_max":   ck["t_max"],
        "y_mean": ck.get("y_mean", 0.0), 
        "y_std":  ck.get("y_std",  1.0),
        "S_Q_all": S_Q_all, 
        "S_BC":    S_BC,
        "S_T_all": S_T_all,
        "meta":    ck["meta"],
    }
    return m, stats


def _norm_branch_for(stats, bQ_case, bBC_case, bT_case):
    bQn  = (bQ_case[None, :]  - stats["bQ_mean"])  / (stats["bQ_std"]  + 1e-8)  # [1, S_Q_all]
    bBCn = (bBC_case[None, :] - stats["bBC_mean"]) / (stats["bBC_std"] + 1e-8)  # [1, S_BC]
    bTn  = (bT_case[None, :]  - stats["bT_mean"])  / (stats["bT_std"]  + 1e-8)  # [1, S_T_all]
    return (bQn.astype(np.float32),
            bBCn.astype(np.float32),
            bTn.astype(np.float32))


def _norm_xy_for(stats, xy):
    return ((xy - stats["xy_min"]) / np.maximum(stats["xy_max"] - stats["xy_min"], 1e-6)).astype(np.float32)


def _norm_t_for(stats, tt):
    return ((tt - stats["t_min"]) / max(stats["t_max"] - stats["t_min"], 1e-6)).astype(np.float32)


def _denorm_y_for(stats, y_norm_tensor):
    # tensor -> physical fraction (tensor)
    return y_norm_tensor * stats["y_std"] + stats["y_mean"]


def _upsample_nn(coarse: np.ndarray, N: int) -> np.ndarray:
    """Nearest-neighbor upsample from SxS -> NxN (no external deps)."""
    S = coarse.shape[0]
    if S == N:
        return coarse.astype(np.float32)
    xi = (np.linspace(0, S-1, N)).round().astype(int)
    yi = (np.linspace(0, S-1, N)).round().astype(int)
    return coarse[np.ix_(yi, xi)].astype(np.float32)


def _reconstruct_Q_map_from_branchQ(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Our branch_Q is concatenated across Nt snapshots.
    For static Q, any one block (per-time length) is fine. Use the FIRST block.
    """
    N   = int(meta_local["N"])
    Nt  = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")
    bQ_all = branch_Q[case_id]  # [S_Q_all]

    if mode == "full":
        per_time = N * N
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length N*N."
        return bQ_all[:per_time].reshape(N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        per_time = S * S
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length S_down*S_down."
        coarse = bQ_all[:per_time].reshape(S, S)
        return _upsample_nn(coarse, N)


def _reconstruct_true_frac_maps(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Rebuild true liquid-fraction maps from (trunk_xy, trunk_t, yf, case_ids)
    assuming FULL grid (sensor_mode == 'full', points_per_case_per_time = N*N).

    We use the (x,y) coordinates to place each yf value onto the correct
    grid cell, rather than naive reshape.
    """
    N      = int(meta_local["N"])
    times  = np.array(meta_local["save_times"], dtype=float)
    Nt     = len(times)

    # select all points for this case
    m_case  = (case_ids == case_id)
    xy_case = trunk_xy[m_case]          # [Nt*N*N, 2]
    t_case  = trunk_t[m_case, 0]        # [Nt*N*N]
    f_case  = yf[m_case, 0]             # [Nt*N*N]

    frac_all = []

    # Loop over each save time
    for t in times:
        m_t  = np.isclose(t_case, t)
        xy_t = xy_case[m_t]             # [N*N, 2]
        f_t  = f_case[m_t]              # [N*N]

        # Sanity check: if we really have a full grid
        assert xy_t.shape[0] == N * N, (
            f"Expected {N*N} points for case {case_id}, t={t}, "
            f"got {xy_t.shape[0]}"
        )

        # Get unique sorted x and y coordinates (with rounding for safety)
        x_vals = np.unique(np.round(xy_t[:, 0], 12))
        y_vals = np.unique(np.round(xy_t[:, 1], 12))
        assert len(x_vals) == N and len(y_vals) == N, \
            f"Unexpected unique x/y counts: {len(x_vals)}, {len(y_vals)}"

        # Map each point to (ix, iy) index on the grid
        x_idx = np.searchsorted(x_vals, np.round(xy_t[:, 0], 12))
        y_idx = np.searchsorted(y_vals, np.round(xy_t[:, 1], 12))

        # Fill grid: row index = y, col index = x
        grid = np.empty((N, N), dtype=np.float32)
        grid[y_idx, x_idx] = f_t

        frac_all.append(grid)

    return np.stack(frac_all, axis=0)    # [Nt, N, N]


@torch.no_grad()
def predict_case_maps(ckpt: Path, case_id: int, sel_times: np.ndarray):
    model, stats = load_trained(ckpt)
    meta_local = stats["meta"]
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])

    # branch vecs for this case (Q, BC, T features)
    bQ_case  = branch_Q[case_id]    # [S_Q_all]
    bBC_case = branch_BC[case_id]   # [S_BC]
    bT_case  = branch_T[case_id]    # [S_T_all]
    bQn, bBCn, bTn = _norm_branch_for(stats, bQ_case, bBC_case, bT_case)

    bQ_t  = torch.from_numpy(bQn).to(device)   # [1, S_Q_all]
    bBC_t = torch.from_numpy(bBCn).to(device)  # [1, S_BC]
    bT_t  = torch.from_numpy(bTn).to(device)   # [1, S_T_all]

    # grid (N x N)
    xs = np.linspace(0.0, Lx, N, dtype=np.float32)
    ys = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    XY = np.stack([X, Y], axis=-1).reshape(-1, 2)  # [N*N, 2]
    XY_n = _norm_xy_for(stats, XY)                 # [N*N, 2]
    XY_t = torch.from_numpy(XY_n).to(device)

    outs = []
    for t in sel_times:
        tt = np.full((XY.shape[0], 1), float(t), dtype=np.float32)  # [N*N, 1]
        tt_n = _norm_t_for(stats, tt)                               # [N*N, 1]
        tt_t = torch.from_numpy(tt_n).to(device)

        # tile branch to match grid points
        B = XY_t.shape[0]
        bQ_tile  = bQ_t.repeat(B, 1)    # [B, S_Q_all]
        bBC_tile = bBC_t.repeat(B, 1)   # [B, S_BC]
        bT_tile  = bT_t.repeat(B, 1)    # [B, S_T_all]

        # model outputs normalized fraction → denormalize
        yhat_n = model(bQ_tile, bBC_tile, bT_tile, XY_t, tt_t)      # [B,1] normalized
        yhat_f = _denorm_y_for(stats, yhat_n).cpu().numpy()         # [B,1] fraction
        outs.append(yhat_f.reshape(N, N).astype(np.float32))
    return np.stack(outs, axis=0)  # [Tsel, N, N]


def _pick_times(times_all: np.ndarray, num_cols: int = 5, prefer: np.ndarray | None = None):
    """Pick ~evenly spaced time stamps (or nearest to 'prefer' if provided)."""
    times_all = np.array(times_all, dtype=float)
    if prefer is None:
        if len(times_all) <= num_cols:
            return times_all
        idx = np.linspace(0, len(times_all)-1, num_cols).round().astype(int)
        return times_all[idx]
    # map preferred to nearest in times_all
    out = []
    for t in prefer:
        out.append(times_all[np.argmin(np.abs(times_all - t))])
    # keep unique in order
    uniq = []
    for t in out:
        if len(uniq) == 0 or abs(uniq[-1] - t) > 1e-12:
            uniq.append(t)
    return np.array(uniq[:num_cols], dtype=float)


def plot_case_transient(ckpt_path: Path, case_id: int, num_cols: int = 5, prefer_times=None):
    _, stats   = load_trained(ckpt_path)
    meta_local = stats["meta"]  # from DATA_FRAC
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])
    times_all = np.array(meta_local["save_times"], dtype=float)
    sel_times = _pick_times(times_all, num_cols=num_cols, prefer=prefer_times)

    # data: Q map, true fraction maps, predicted fraction maps
    Q_map        = _reconstruct_Q_map_from_branchQ(case_id, meta_local)      # [N,N]
    frac_true_all = _reconstruct_true_frac_maps(case_id, meta_local)         # [Nt,N,N]
    idx_true      = np.array([np.argmin(np.abs(times_all - t)) for t in sel_times], dtype=int)
    frac_true     = frac_true_all[idx_true]                                  # [Tsel,N,N]
    frac_pred     = predict_case_maps(ckpt_path, case_id, sel_times)         # [Tsel,N,N]

    # color scaling for fraction (0–1-ish)
    vmin = min(frac_true.min(), frac_pred.min())
    vmax = max(frac_true.max(), frac_pred.max())

    cols = len(sel_times)
    fig, axes = plt.subplots(nrows=3, ncols=cols, figsize=(3.2*cols, 9.0), constrained_layout=True)

    # Row 0: Heat source (repeat same Q for alignment)
    for c in range(cols):
        ax = axes[0, c]
        imQ = ax.imshow(Q_map.T, origin="lower", extent=[0, Lx, 0, Ly], cmap="RdBu_r", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Heat source Q", fontsize=11)
        ax.set_title(f"t = {sel_times[c]:g}s", fontsize=11)
    fig.colorbar(imQ, ax=axes[0, :].ravel().tolist(), fraction=0.02, pad=0.02)

    # Row 1: True fraction
    for c in range(cols):
        ax = axes[1, c]
        imT = ax.imshow(frac_true[c].T, origin="lower", extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("True fraction", fontsize=11)
    fig.colorbar(imT, ax=axes[1, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("f_liq")

    # Row 2: Predicted fraction
    for c in range(cols):
        ax = axes[2, c]
        imP = ax.imshow(frac_pred[c].T, origin="lower", extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Pred fraction", fontsize=11)
    fig.colorbar(imP, ax=axes[2, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("f_liq")

    plt.show()

In [ ]:
RUN_DIR   = Path(r"runs/dataset_run_20251120-131248")
DATA_TEMP = np.load(RUN_DIR / "deeponet_temp_dataset.npz", allow_pickle=True)
DATA_FRAC = np.load(RUN_DIR / "deeponet_frac_dataset.npz", allow_pickle=True)

branch_Q  = DATA_TEMP["branch_Q"]
branch_BC = DATA_TEMP["branch_BC"]

branch_T  = DATA_FRAC["branch_T"]
branch_F  = DATA_FRAC["branch_F"]   
meta      = DATA_FRAC["meta"].item()
save_times = np.array(meta["save_times"], dtype=float)

In [ ]:
# -------------------------------
# 1) LOAD TRAINED FRAC MODEL
# -------------------------------

@torch.no_grad()
def load_trained_frac(path: Path):
    ck = torch.load(path, map_location="cpu", weights_only=False)

    S_Q_all = ck["S_Q_all"]
    S_BC    = ck["S_BC"]
    S_T_all = ck["S_T_all"]

    # Use stored config if present, else fall back to train defaults
    cfg = ck.get("model_config", None)
    if cfg is None:
        cfg = dict(
            D=256,
            Q_hidden=(256,256,256),
            BC_hidden=None,
            T_hidden=(256,256,256),
            xy_hidden=(256,256,256,256),
            t_hidden=(128,128),
        )

    m = DeepONet5(
        S_Q_all=S_Q_all,
        S_BC=S_BC,
        S_T_all=S_T_all,
        **cfg
    ).to(device)

    m.load_state_dict(ck["model"], strict=True)
    m.eval()

    stats = {
        "bQ_mean": ck["bQ_mean"],  "bQ_std":  ck["bQ_std"],
        "bBC_mean": ck["bBC_mean"], "bBC_std": ck["bBC_std"],
        "bT_mean": ck["bT_mean"],  "bT_std":  ck["bT_std"],
        "xy_min": ck["xy_min"],    "xy_max":  ck["xy_max"],
        "t_min": ck["t_min"],      "t_max":   ck["t_max"],
        "y_mean": ck.get("y_mean", 0.0),
        "y_std":  ck.get("y_std",  1.0),
        "meta":   ck["meta"],
        "S_Q_all": S_Q_all,
        "S_BC":    S_BC,
        "S_T_all": S_T_all,
    }
    return m, stats


In [ ]:
# -------------------------------
# 2) HELPERS
# -------------------------------

def _norm_branch_frac(stats, bQ_case, bBC_case, bT_case):
    bQn  = (bQ_case[None,:]  - stats["bQ_mean"])  / (stats["bQ_std"]  + 1e-8)
    bBCn = (bBC_case[None,:] - stats["bBC_mean"]) / (stats["bBC_std"] + 1e-8)
    bTn  = (bT_case[None,:]  - stats["bT_mean"])  / (stats["bT_std"]  + 1e-8)
    return (bQn.astype(np.float32),
            bBCn.astype(np.float32),
            bTn.astype(np.float32))

def _norm_xy(stats, XY):
    return ((XY - stats["xy_min"]) /
            np.maximum(stats["xy_max"] - stats["xy_min"], 1e-6)
           ).astype(np.float32)

def _norm_t(stats, tt):
    return ((tt - stats["t_min"]) /
            max(stats["t_max"] - stats["t_min"], 1e-6)
           ).astype(np.float32)

def _denorm_y(stats, yn):
    return yn * stats["y_std"] + stats["y_mean"]

def _upsample_nn(coarse: np.ndarray, N: int) -> np.ndarray:
    S = coarse.shape[0]
    if S == N:
        return coarse.astype(np.float32)
    xi = (np.linspace(0, S-1, N)).round().astype(int)
    yi = (np.linspace(0, S-1, N)).round().astype(int)
    return coarse[np.ix_(yi, xi)].astype(np.float32)

def _reconstruct_Q_map_from_branchQ(case_id: int, meta_local: dict) -> np.ndarray:
    N    = int(meta_local["N"])
    Nt   = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")
    bQ_all = branch_Q[case_id]

    if mode == "full":
        per_time = N * N
        assert bQ_all.shape[0] == Nt * per_time
        return bQ_all[:per_time].reshape(N, N).astype(np.float32)
    else:
        S = int(meta_local["S_down"])
        per_time = S * S
        assert bQ_all.shape[0] == Nt * per_time
        coarse = bQ_all[:per_time].reshape(S, S)
        return _upsample_nn(coarse, N)

def _reconstruct_true_frac_from_branchF(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Use branch_F (full fraction fields) to make [Nt, N, N] true maps.
    """
    N    = int(meta_local["N"])
    Nt   = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")

    Fvec = branch_F[case_id]   # [Nt * N*N] or [Nt * S*S]

    if mode == "full":
        return Fvec.reshape(Nt, N, N).astype(np.float32)
    else:
        S = int(meta_local["S_down"])
        coarse = Fvec.reshape(Nt, S, S)
        return np.stack([_upsample_nn(coarse[k], N) for k in range(Nt)],
                        axis=0).astype(np.float32)


In [ ]:
# -------------------------------
# 3) PREDICT FRACTION MAPS
# -------------------------------

@torch.no_grad()
def predict_case_frac(ckpt_path: Path, case_id: int, sel_times: np.ndarray):
    model, stats = load_trained_frac(ckpt_path)
    meta_local   = stats["meta"]

    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])

    # branch vectors for this case
    bQ_case  = branch_Q[case_id]
    bBC_case = branch_BC[case_id]
    bT_case  = branch_T[case_id]
    bQn, bBCn, bTn = _norm_branch_frac(stats, bQ_case, bBC_case, bT_case)

    bQ_t  = torch.from_numpy(bQn).to(device)
    bBC_t = torch.from_numpy(bBCn).to(device)
    bT_t  = torch.from_numpy(bTn).to(device)

    # grid
    xs = np.linspace(0.0, Lx, N, dtype=np.float32)
    ys = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    XY   = np.stack([X, Y], axis=-1).reshape(-1, 2)
    XY_n = _norm_xy(stats, XY)
    XY_t = torch.from_numpy(XY_n).to(device)

    outs = []
    for t in sel_times:
        tt   = np.full((XY.shape[0], 1), float(t), dtype=np.float32)
        tt_n = _norm_t(stats, tt)
        tt_t = torch.from_numpy(tt_n).to(device)

        B = XY.shape[0]
        bQ_tile  = bQ_t.repeat(B, 1)
        bBC_tile = bBC_t.repeat(B, 1)
        bT_tile  = bT_t.repeat(B, 1)

        yhat_n = model(bQ_tile, bBC_tile, bT_tile, XY_t, tt_t)
        yhat_f = _denorm_y(stats, yhat_n).cpu().numpy()

        outs.append(yhat_f.reshape(N, N).astype(np.float32))

    return np.stack(outs, axis=0)   # [Tsel, N, N]


In [ ]:
# -------------------------------
# 4) TIME PICKER
# -------------------------------

def _pick_times(times_all: np.ndarray, num_cols: int = 5, prefer=None):
    times_all = np.array(times_all, dtype=float)
    if prefer is None:
        if len(times_all) <= num_cols:
            return times_all
        idx = np.linspace(0, len(times_all)-1, num_cols).round().astype(int)
        return times_all[idx]
    out = [times_all[np.argmin(np.abs(times_all - t))] for t in prefer]
    uniq = []
    for t in out:
        if not uniq or abs(uniq[-1] - t) > 1e-12:
            uniq.append(t)
    return np.array(uniq[:num_cols], dtype=float)


In [ ]:
# -------------------------------
# 5) MAIN PLOT FUNCTION
# -------------------------------

def plot_case_transient_frac(ckpt_path: Path, case_id: int,
                             num_cols: int = 5, prefer_times=None):
    # meta from checkpoint (to be safe)
    _, stats   = load_trained_frac(ckpt_path)
    meta_local = stats["meta"]

    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])
    times_all = np.array(meta_local["save_times"], dtype=float)

    sel_times = _pick_times(times_all, num_cols=num_cols, prefer=prefer_times)

    Q_map         = _reconstruct_Q_map_from_branchQ(case_id, meta_local)
    frac_true_all = _reconstruct_true_frac_from_branchF(case_id, meta_local)
    idx_true      = np.array([np.argmin(np.abs(times_all - t)) for t in sel_times],
                             dtype=int)
    frac_true     = frac_true_all[idx_true]
    frac_pred     = predict_case_frac(ckpt_path, case_id, sel_times)

    vmin = min(frac_true.min(), frac_pred.min())
    vmax = max(frac_true.max(), frac_pred.max())

    cols = len(sel_times)
    fig, axes = plt.subplots(nrows=3, ncols=cols,
                             figsize=(3.2*cols, 9.0),
                             constrained_layout=True)

    # Row 0: Q
    for c in range(cols):
        ax = axes[0, c]
        imQ = ax.imshow(Q_map.T, origin="lower",
                        extent=[0, Lx, 0, Ly],
                        cmap="RdBu_r", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Heat source Q", fontsize=11)
        ax.set_title(f"t = {sel_times[c]:g}s", fontsize=11)
    fig.colorbar(imQ, ax=axes[0, :].ravel().tolist(),
                 fraction=0.02, pad=0.02)

    # Row 1: true f_liq
    for c in range(cols):
        ax = axes[1, c]
        imT = ax.imshow(frac_true[c].T, origin="lower",
                        extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax,
                        aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("True fraction", fontsize=11)
    fig.colorbar(imT, ax=axes[1, :].ravel().tolist(),
                 fraction=0.02, pad=0.02).set_label("f_liq")

    # Row 2: pred f_liq
    for c in range(cols):
        ax = axes[2, c]
        imP = ax.imshow(frac_pred[c].T, origin="lower",
                        extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax,
                        aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Pred fraction", fontsize=11)
    fig.colorbar(imP, ax=axes[2, :].ravel().tolist(),
                 fraction=0.02, pad=0.02).set_label("f_liq")

    plt.show()


In [ ]:
# Pick a demo case and plot
ckpt_path = Path(r'runs/dataset_run_20251120-131248/deeponet_frac_model_5net.pt')

for i in range(len(test_ids)):
    demo_case = int(test_ids[i])  # 0-based indexing
    plot_case_transient_frac(ckpt_path, demo_case, num_cols=7, prefer_times=save_times)
    print(demo_case + 1)          # print as 1-based for readability